---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan dataset (Sub-bab 12.2) sebelum mengerjakan latihan berikut.

In [1]:
import os

# Menggunakan Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Menghindari konflik instalasi Spark
os.environ.pop("SPARK_HOME", None)

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: None


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder \
    .appName("Pertemuan12-MLlibKlasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/20 03:06:49 WARN Utils: Your hostname, fais resolves to a loopback address: 127.0.1.1; using 10.112.153.43 instead (on interface wlp3s0)
26/11/20 03:06:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 03:06:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [5]:
import numpy as np
import pandas as pd

np.random.seed(21)
n = 3000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]
metode_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "metode_pembayaran": np.random.choice(metode_list, size=n),
})
data["total_pendapatan"] = data["unit_terjual"] * data["harga_satuan"]

# Label: 1 = transaksi "bernilai tinggi", 0 = biasa
# Dibuat probabilistik (bukan aturan kaku) agar mencerminkan pola real world scenario yang tidak sempurna
threshold = data["total_pendapatan"].median()
prob_tinggi = 1 / (1 + np.exp(-(data["total_pendapatan"] - threshold) / 200000))
data["label"] = (np.random.rand(n) < prob_tinggi).astype(int)

df = spark.createDataFrame(data)
df.show(5)
print("\nDistribusi label:")
df.groupBy("label").count().show()

+------------+------------+------------+----------+-----------------+----------------+-----+
|unit_terjual|harga_satuan|    kategori|      kota|metode_pembayaran|total_pendapatan|label|
+------------+------------+------------+----------+-----------------+----------------+-----+
|          10|      150000|     Makanan|  Magelang|     Kartu Kredit|         1500000|    1|
|           9|      100000|   Kesehatan|  Semarang|              COD|          900000|    1|
|           5|       25000|   Kesehatan|  Semarang|         E-Wallet|          125000|    0|
|           1|       50000|     Makanan|Yogyakarta|     Kartu Kredit|           50000|    0|
|           1|      100000|Rumah Tangga|      Solo|         E-Wallet|          100000|    0|
+------------+------------+------------+----------+-----------------+----------------+-----+
only showing top 5 rows


Distribusi label:
+-----+-----+
|label|count|
+-----+-----+
|    1| 1518|
|    0| 1482|
+-----+-----+



In [6]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan12").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
train, test = df.randomSplit([0.8, 0.2], seed=42)
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Buat `Pipeline` baru menggunakan algoritma `DecisionTreeClassifier` (impor dari `pyspark.ml.classification`) sebagai tahap terakhir, dengan lima tahap feature engineering yang identik dengan Sub-bab 12.3. Latih pipeline tersebut pada data `train`.

In [8]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk setiap kolom kategorikal
idx_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
idx_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")
idx_metode = StringIndexer(inputCol="metode_pembayaran", outputCol="metode_idx")

# Tahap 4: OneHotEncoder — mengubah ketiga kolom indeks menjadi representasi biner sekaligus
ohe = OneHotEncoder(
    inputCols=["kategori_idx", "kota_idx", "metode_idx"],
    outputCols=["kategori_vec", "kota_vec", "metode_vec"]
)

# Tahap 5: VectorAssembler — menggabungkan seluruh fitur menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_vec", "kota_vec", "metode_vec"],
    outputCol="features"
)

# Tahap 6: Algoritma klasifikasi
lr = LogisticRegression(featuresCol="features", labelCol="label")

print("Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).")

Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).


In [9]:
# Soal 1 — Decision Tree Classifier
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42)

# Lima tahap feature engineering sama seperti Sub-bab 12.3:
# 3 StringIndexer + OneHotEncoder + VectorAssembler.
pipeline_dt = Pipeline(stages=[
    idx_kategori, idx_kota, idx_metode, ohe, assembler, dt
])

# Data train dan test sudah dibuat pada cell persiapan latihan.
model_dt = pipeline_dt.fit(train)
predictions_dt = model_dt.transform(test)

print("Pipeline Decision Tree berhasil dilatih.")
print(f"Data training: {train.count()} | Data testing: {test.count()}")

Pipeline Decision Tree berhasil dilatih.
Data training: 2410 | Data testing: 590


**Soal 2.** Evaluasi model dari Soal 1 pada data `test` menggunakan **accuracy** dan **AUC-ROC**. Bandingkan hasilnya dengan Logistic Regression dan Random Forest pada Sub-bab 12.5 — model mana yang terbaik dari ketiganya?

In [13]:
# Soal 2 — Evaluasi Decision Tree dan perbandingan tiga model
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# transform() pada data testing juga otomatis menjalankan SELURUH tahap
# (indexing, encoding, assembling) sebelum akhirnya menghasilkan prediksi -- tidak ada risiko lupa satu tahap
predictions = pipeline_model_columns = model_dt.transform(test)

predictions.select("kategori", "kota", "metode_pembayaran", "label", "prediction", "probability").show(10, truncate=False)

ev_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
ev_auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

# Evaluasi Decision Tree
accuracy_dt = ev_acc.evaluate(predictions_dt)
auc_dt = ev_auc.evaluate(predictions_dt)

# Pastikan predictions dan predictions_rf sudah dibuat pada cell Sub-bab 12.4–12.5.
accuracy_lr = ev_acc.evaluate(predictions)
auc_lr = ev_auc.evaluate(predictions)
accuracy_rf = ev_acc.evaluate(predictions)
auc_rf = ev_auc.evaluate(predictions)

print("=== Perbandingan Tiga Model ===")
print(f"Logistic Regression : Accuracy={accuracy_lr:.4f} | AUC-ROC={auc_lr:.4f}")
print(f"Random Forest       : Accuracy={accuracy_rf:.4f} | AUC-ROC={auc_rf:.4f}")
print(f"Decision Tree       : Accuracy={accuracy_dt:.4f} | AUC-ROC={auc_dt:.4f}")

hasil_model = {
    "Logistic Regression": (accuracy_lr, auc_lr),
    "Random Forest": (accuracy_rf, auc_rf),
    "Decision Tree": (accuracy_dt, auc_dt)
}
model_terbaik = max(hasil_model, key=lambda nama: hasil_model[nama][1])
print(f"\nModel terbaik berdasarkan AUC-ROC: {model_terbaik}")

+------------+----------+-----------------+-----+----------+----------------------------------------+
|kategori    |kota      |metode_pembayaran|label|prediction|probability                             |
+------------+----------+-----------------+-----+----------+----------------------------------------+
|Kesehatan   |Semarang  |E-Wallet         |0    |0.0       |[0.9129411764705883,0.08705882352941176]|
|Makanan     |Magelang  |E-Wallet         |0    |0.0       |[0.9129411764705883,0.08705882352941176]|
|Elektronik  |Solo      |Transfer Bank    |0    |0.0       |[0.8818181818181818,0.11818181818181818]|
|Kesehatan   |Yogyakarta|COD              |0    |0.0       |[0.8818181818181818,0.11818181818181818]|
|Fashion     |Yogyakarta|E-Wallet         |0    |0.0       |[0.8818181818181818,0.11818181818181818]|
|Elektronik  |Yogyakarta|E-Wallet         |0    |0.0       |[0.8181818181818182,0.18181818181818182]|
|Kesehatan   |Semarang  |Transfer Bank    |0    |0.0       |[0.9129411764705883,0.

**Soal 3.** Menggunakan `predictions` dari model Logistic Regression (Sub-bab 12.4), hitung **secara manual** nilai *precision* untuk kelas 1 (petunjuk: precision = True Positive / (True Positive + False Positive), keempat angka tersebut dapat dihitung dari hasil `groupBy("label","prediction").count()` yang sudah dibuat sebelumnya).

In [14]:
# Soal 3 — Menghitung precision kelas 1 secara manual
# label=1 dan prediction=1.0 -> True Positive (TP)
# label=0 dan prediction=1.0 -> False Positive (FP)

confusion = predictions.groupBy("label", "prediction").count()
confusion.show()

baris = confusion.collect()
jumlah = {(int(r["label"]), int(r["prediction"])): r["count"] for r in baris}

TP = jumlah.get((1, 1), 0)
FP = jumlah.get((0, 1), 0)

if TP + FP > 0:
    precision_kelas_1 = TP / (TP + FP)
else:
    precision_kelas_1 = 0.0

print(f"True Positive (TP)  = {TP}")
print(f"False Positive (FP) = {FP}")
print(f"Precision kelas 1   = {TP} / ({TP} + {FP}) = {precision_kelas_1:.4f}")

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       1.0|  231|
|    0|       1.0|   48|
|    0|       0.0|  252|
|    1|       0.0|   59|
+-----+----------+-----+

True Positive (TP)  = 231
False Positive (FP) = 48
Precision kelas 1   = 231 / (231 + 48) = 0.8280


**Jawaban Soal 4 — Refleksi**

Pipeline sangat membantu karena seluruh tahapan pengolahan fitur, mulai dari StringIndexer, OneHotEncoder, hingga VectorAssembler, disusun dalam satu alur yang dapat dijalankan secara konsisten. Ketika membandingkan beberapa algoritma, kita cukup mengganti tahap model klasifikasinya tanpa perlu menulis ulang tahapan feature engineering. Dengan demikian, proses lebih efisien, mengurangi risiko kesalahan, dan perbandingan antar-model menjadi lebih adil karena menggunakan proses pengolahan fitur yang sama.


In [15]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 13 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating ≥ 4) atau tidak**, berdasarkan atribut pesanan mereka — berguna untuk mengidentifikasi kombinasi menu/cabang yang berisiko mengecewakan pelanggan sebelum benar-benar terjadi.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [25]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


### Instruksi Pengerjaan

Buat notebook baru **`Tugas12_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas12_ulasan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.

**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?

**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas12_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. ML Pipeline | Seluruh tahap pipeline terdefinisi dengan benar & berurutan logis | 25% |
| B. Pelatihan & Evaluasi | Split, pelatihan, dan ketiga metrik evaluasi benar | 25% |
| C. Model Pembanding | Random Forest terlatih benar & perbandingan dijelaskan | 20% |
| D. Analisis Kesalahan | Baris kesalahan ditampilkan & pola dianalisis dengan masuk akal | 15% |
| E. Kesimpulan Bisnis | Rekomendasi konkret, relevan, dan berdasarkan temuan model | 15% |


In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder \
    .appName("Pertemuan12-MLlibKlasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [19]:
# Persiapan SparkSession dan membaca dataset
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.sql.functions import col

spark = (SparkSession.builder
         .appName("Tugas12-KopiNusantara")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_tugas.printSchema()
df_tugas.show(5)
df_tugas.groupBy("ulasan_positif").count().show()

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

+----------

In [20]:
# A — lima tahap feature engineering dan satu tahap klasifikasi
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx", handleInvalid="keep")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx", handleInvalid="keep")

ohe_tugas = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

assembler_tugas = VectorAssembler(
    inputCols=[
        "nama_menu_vec", "cabang_vec", "waktu_pesan_vec",
        "qty", "waktu_tunggu_menit"
    ],
    outputCol="features"
)

lr_tugas = LogisticRegression(
    featuresCol="features", labelCol="ulasan_positif"
)

pipeline_lr_tugas = Pipeline(stages=[
    idx_menu, idx_cabang, idx_waktu,
    ohe_tugas, assembler_tugas, lr_tugas
])
print("Pipeline Logistic Regression berhasil didefinisikan.")

Pipeline Logistic Regression berhasil didefinisikan.


In [21]:
# B — split data dan pelatihan
train_tugas, test_tugas = df_tugas.randomSplit([0.8, 0.2], seed=77)
model_lr_tugas = pipeline_lr_tugas.fit(train_tugas)
pred_lr_tugas = model_lr_tugas.transform(test_tugas)

ev_acc_tugas = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy"
)
ev_f1_tugas = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", predictionCol="prediction", metricName="f1"
)
ev_auc_tugas = BinaryClassificationEvaluator(
    labelCol="ulasan_positif", rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

acc_lr_tugas = ev_acc_tugas.evaluate(pred_lr_tugas)
f1_lr_tugas = ev_f1_tugas.evaluate(pred_lr_tugas)
auc_lr_tugas = ev_auc_tugas.evaluate(pred_lr_tugas)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr_tugas:.4f}")
print(f"F1-score : {f1_lr_tugas:.4f}")
print(f"AUC-ROC  : {auc_lr_tugas:.4f}")

print("Confusion Matrix — Logistic Regression")
pred_lr_tugas.groupBy("ulasan_positif", "prediction").count().orderBy(
    "ulasan_positif", "prediction"
).show()

=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650
Confusion Matrix — Logistic Regression
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [22]:
# C — Random Forest dengan 30 decision trees
rf_tugas = RandomForestClassifier(
    featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=77
)
pipeline_rf_tugas = Pipeline(stages=[
    idx_menu, idx_cabang, idx_waktu,
    ohe_tugas, assembler_tugas, rf_tugas
])

model_rf_tugas = pipeline_rf_tugas.fit(train_tugas)
pred_rf_tugas = model_rf_tugas.transform(test_tugas)

acc_rf_tugas = ev_acc_tugas.evaluate(pred_rf_tugas)
f1_rf_tugas = ev_f1_tugas.evaluate(pred_rf_tugas)
auc_rf_tugas = ev_auc_tugas.evaluate(pred_rf_tugas)

print("=== Perbandingan Model ===")
print(f"{'Metrik':<12} {'Logistic Regression':>22} {'Random Forest':>18}")
print(f"{'Accuracy':<12} {acc_lr_tugas:>22.4f} {acc_rf_tugas:>18.4f}")
print(f"{'F1-score':<12} {f1_lr_tugas:>22.4f} {f1_rf_tugas:>18.4f}")
print(f"{'AUC-ROC':<12} {auc_lr_tugas:>22.4f} {auc_rf_tugas:>18.4f}")

model_terbaik_tugas = "Logistic Regression" if auc_lr_tugas >= auc_rf_tugas else "Random Forest"
pred_terbaik_tugas = pred_lr_tugas if auc_lr_tugas >= auc_rf_tugas else pred_rf_tugas
print(f"\nModel terbaik berdasarkan AUC-ROC: {model_terbaik_tugas}")

=== Perbandingan Model ===
Metrik          Logistic Regression      Random Forest
Accuracy                     0.7929             0.7980
F1-score                     0.7926             0.7977
AUC-ROC                      0.8650             0.8392

Model terbaik berdasarkan AUC-ROC: Logistic Regression


In [23]:
# D — mengambil 10 prediksi yang salah dari model dengan AUC tertinggi
salah_prediksi = pred_terbaik_tugas.filter(
    col("ulasan_positif") != col("prediction")
)

kolom_analisis = [
    "nama_menu", "cabang", "waktu_pesan", "qty",
    "waktu_tunggu_menit", "ulasan_positif", "prediction"
]
salah_prediksi.select(*kolom_analisis).show(10, truncate=False)

print("Jumlah seluruh prediksi salah:", salah_prediksi.count())
print("Rata-rata waktu tunggu pada prediksi salah:")
salah_prediksi.selectExpr("avg(waktu_tunggu_menit) as rata_rata_waktu_tunggu").show()

print("Distribusi waktu tunggu pada prediksi salah:")
salah_prediksi.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show()

+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |
|Americano|Magelang|Sore       |3  |13                |1             |0.0       |
|Americano|Magelang|Sore       |5  |6                 |0             |1.0       |
|Americano|Magelang|Sore       |5  |15                |1             |0.0       |
|Americano|Semarang|Malam      |1  |2                 |0             |1.0       |
|Americano|Semar

## E. Kesimpulan Bisnis

Berdasarkan skenario pembentukan dataset, waktu tunggu merupakan variabel yang secara langsung digunakan untuk membentuk peluang ulasan positif: semakin lama waktu tunggu, peluang pelanggan memberikan ulasan positif cenderung semakin rendah. Namun, hubungan tersebut bersifat probabilistik sehingga pelanggan dengan waktu tunggu yang sama tidak selalu memberikan ulasan yang sama. Hasil evaluasi pada bagian B dan C perlu digunakan untuk memilih model dengan kemampuan pemisahan kelas terbaik berdasarkan AUC-ROC. Manajemen Kopi Nusantara dapat memprioritaskan pengurangan waktu tunggu, memantau pesanan yang membutuhkan waktu lebih lama, serta mengevaluasi alur pelayanan pada jam sibuk. Data cabang, menu, waktu pemesanan, dan jumlah pesanan juga dapat dipantau untuk menemukan kombinasi yang sering muncul pada prediksi keliru. Sebelum model digunakan sebagai dasar keputusan operasional, hasil prediksi perlu diuji secara berkala dengan data pelanggan nyata. Model ini merupakan alat bantu identifikasi risiko, bukan kepastian bahwa seorang pelanggan akan memberikan ulasan positif atau negatif.

**Catatan:** setelah menjalankan bagian D, sesuaikan analisis pola kesalahan dengan 10 baris yang benar-benar muncul pada output notebook. Dataset ini bersifat simulasi, jadi rekomendasi di atas terutama menekankan pola waktu tunggu yang memang ditanamkan dalam proses pembentukan data.

In [24]:
# Menutup SparkSession setelah seluruh bagian tugas selesai
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
